In [1]:
!pip install transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 26.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

In [2]:
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, AutoConfig
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import evaluate
import json

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_csv("/content/drive/MyDrive/SIC/Dataset/data_for_ml.csv", encoding='utf-8-sig')
df = df[['segment_text', 'label']]

In [5]:
# Load mapping từ JSON
with open("/content/drive/MyDrive/SIC/Mapping/label_mapping.json", "r", encoding="utf-8") as f:
    mapping_data = json.load(f)

int_to_label= mapping_data["int_to_label"]

# Chuyển label thành số từ label mapping
label_to_int = mapping_data["label_to_int"]

## Tạo cột label_int
df["label_int"] = df["label"].map(label_to_int)


In [6]:
df

,segment_text,label,label_int
0,shop phục_vụ rất kém,neg,0
1,"tôi về vất đi rồi , chỉ được quảng_cáo hay thô...",neg,0
2,chất_lượng sản_phẩm rất kém đóng_gói sản_phẩm ...,neg,0
3,bề ngang áo chật,neg,0
4,và hẳn dây thì tai mèo,neg,0
...,...,...,...
14967,mỗi mềm hơn rất nhju,pos,2
14968,cực_kì đáng tiền,pos,2
14969,"rẻ , đẹp_trai",pos,2
14970,"chất_lượng , màu_sắc khá ổn , giống ảnh . khá ...",pos,2


In [7]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label_int"], random_state=42)

# 3. Chia tập còn lại thành validation (15%) và test (15%)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label_int"], random_state=42)

# 4. Chuyển thành Dataset của HuggingFace
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# 5. Gom thành một DatasetDict (nếu bạn dùng với Trainer)
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

# Tuỳ chọn: reset lại index nếu cần
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [8]:
dataset

DatasetDict({
    train: Dataset({
        features: ['segment_text', 'label', 'label_int', '__index_level_0__'],
        num_rows: 10480
    })
    validation: Dataset({
        features: ['segment_text', 'label', 'label_int', '__index_level_0__'],
        num_rows: 2246
    })
    test: Dataset({
        features: ['segment_text', 'label', 'label_int', '__index_level_0__'],
        num_rows: 2246
    })
})

In [9]:
print(f"Train: {len(train_df)} samples")
print(f"Validation: {len(val_df)} samples")
print(f"Test: {len(test_df)} samples")

Train: 10480 samples
Validation: 2246 samples
Test: 2246 samples


## Tải bộ tokenizer từ HuggingFace, cụ thể là Sentencepiece Tokenizer dùng cho PhoBERT

In [10]:
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Tạo 1 hàm Tokenize cho các câu trong dataset
## Sau đó chuyển các DataFrame của Pandas thành các đối tượng Dataset của thư viện HuggingFace

In [12]:
def tokenize_function(example):

In [13]:
train_dataset = Dataset.from_pandas(train_df[['segment_text', 'label_int']])
val_dataset = Dataset.from_pandas(val_df[['segment_text', 'label_int']])
test_dataset = Dataset.from_pandas(test_df[['segment_text', 'label_int']])

## Token hóa toàn bộ dữ liệu (train, val, test) bằng tokenizer đã định nghĩa.
# Dùng batched=True để xử lý nhanh hơn.

In [14]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)


#Đổi tên label_int về label cho đúng chuẩn HuggingFace Trainer
train_dataset = train_dataset.rename_column("label_int", "labels")
val_dataset = val_dataset.rename_column("label_int", "labels")
test_dataset = test_dataset.rename_column("label_int", "labels")

#Đặt định dạng cho PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/10480 [00:00<?, ? examples/s]

KeyError: 'text'

## TUỲ CHỈNH CẤU HÌNH TỪ MODEL GỐC VÀ ÁP DỤNG NÓ CHO MODEL



In [ ]:
config = AutoConfig.from_pretrained(
    "vinai/phobert-base",  # Tải cấu hình từ model gốc PhoBERT
    num_labels=3,   # Số lớp phân loại (3 nhãn: neg, neu, pos)
    id2label={0: "negative", 1: "neutral", 2: "positive"}, # Ánh xạ ID → tên nhãn
    label2id={"negative": 0, "neutral": 1, "positive": 2}, # Ánh xạ tên nhãn → ID , mục đích của 2 cái này là giúp log dễ hiểu và hỗ trợ metric
    hidden_dropout_prob=0.1,  # Tỷ lệ dropout giữa các lớp hidden (để tránh overfitting)
    attention_probs_dropout_prob=0.1, # Dropout cho attention scores
    finetuning_task="sentiment", # Tên task
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", config=config)

## Cấu hình model

In [ ]:
print(model.config)

## TẠO HÀM ĐÁNH GIÁ MODEL

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted"),
        "precision": precision_score(labels, predictions, average="weighted"),
        "recall": recall_score(labels, predictions, average="weighted"),
    }

## Cấu hình tham số huấn luyện (TrainingArguments)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",                 # Thư mục lưu mô hình sau mỗi epoch.
    eval_strategy="epoch",                 # Đánh giá mô hình sau mỗi epoch.
    save_strategy="epoch",                 # Lưu mô hình sau mỗi epoch.
    learning_rate=2e-5,                    # Learning rate cho optimizer.
    per_device_train_batch_size=16,        # Batch size khi train cho mỗi thiết bị (GPU/CPU).
    per_device_eval_batch_size=32,         # Batch size khi validation.
    num_train_epochs=4,                    # Số epoch huấn luyện.
    weight_decay=0.01,                     # Hệ số regularization để tránh overfitting.
    load_best_model_at_end=True,           # Tự động load mô hình tốt nhất sau khi huấn luyện.
    metric_for_best_model="f1",            # Sử dụng F1-score để chọn mô hình tốt nhất.
    logging_dir="./logs",                  # Thư mục lưu logs cho TensorBoard.
    logging_steps=50,                      # Ghi log sau mỗi 50 bước.
    save_total_limit=1,                    # Chỉ giữ lại 1 checkpoint tốt nhất, xóa các checkpoint cũ hơn.
    report_to="none",                      # Không gửi log đến hệ thống bên ngoài.
)

In [ ]:
!pip install "numpy<2.0"  #tải cái này tại lúc trainer nó không hỗ trợ cho numpy bản mới nhất

## TRAIN VÀ ĐÁNH GIÁ

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
trainer.save_model("/content/drive/MyDrive/SIC/Model/phobert_sentiment_custom_finetuned",safe_serialization=False)
tokenizer.save_pretrained("/content/drive/MyDrive/SIC/Model/phobert_sentiment_custom_finetuned")
print("\nMô hình đã được lưu tại thư mục '/content/drive/MyDrive/SIC/Model/phobert_sentiment_custom_finetuned'")

In [ ]:
print("\n Đánh giá nhanh trên tập test")
test_results = trainer.evaluate(test_dataset)
print(test_results)

from sklearn.metrics import classification_report

preds_output = trainer.predict(test_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
y_true = preds_output.label_ids

print("\n Báo cáo phân loại:")
print(classification_report(y_true, y_pred, target_names=["neg", "neu", "pos"]))


 Đánh giá nhanh trên tập test


{'eval_loss': 0.6873846650123596, 'eval_accuracy': 0.7226179875333927, 'eval_f1': 0.7181727743305576, 'eval_precision': 0.7170578834602743, 'eval_recall': 0.7226179875333927, 'eval_runtime': 13.3255, 'eval_samples_per_second': 168.55, 'eval_steps_per_second': 5.328, 'epoch': 4.0}

 Báo cáo phân loại:
              precision    recall  f1-score   support

         neg       0.74      0.77      0.75       749
         neu       0.64      0.55      0.59       749
         pos       0.77      0.84      0.81       748

    accuracy                           0.72      2246
   macro avg       0.72      0.72      0.72      2246
weighted avg       0.72      0.72      0.72      2246

